In [1]:
import warnings
import cobra

import pandas as pd

import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
from utils import *

load environmental variables
The project root is: /Users/joycebaghdassarian/Documents/UCSD/Lewis_Lab/Projects/human_me/


-To change to individual tRNA molecules rather than a generic one, start with the charge_trna function and also change the trna_biogenesis function

-To add modifications, will need to change the allowed_trna_modifications dictionary in utils, and edit the following functions: 1) modify_trna_nuclear, 2) modify_trna_cytosolic, 3) degrade_trna

# TRNA Information Class

In [3]:
class trna_information():
    def __init__(self, maturetrna_sequence , id_, three_trailer_seq = None , five_leader_seq = None, 
                 modifications = {}, intron_sequences = None):
        '''
        1) Mature trna sequence is the RNA sequence of the final processed tRNA, represented as a string from 5'
        to 3' end, including CCA.
        2) id_ should be map to isodecoder somehow
        2-3) five_leader_seq and three_trailer_seq are the RNA sequences of the 5' leader and 3' trailer sequences that are 
        excised. Represented as a string from 5' to 3' end.
        5) Modifications is a dictionary with keys as possible modifications (string) and values as the 
        number of modifications that occur (integer). See allowed_trna_modifications for all trna modifications
        incorporated in this model. 
        6) Intron sequences is a list of RNA sequence corresponding to each intron. 
        '''
        
        
        if maturetrna_sequence[-3:] != 'CCA':
            warnings.warn('CCA tail not present in provided mature sequence, adding to 3 primed end')
            maturetrna_sequence += 'CCA'
        if len(maturetrna_sequence) < 73 or len(maturetrna_sequence) > 93:
            # https://www.nature.com/articles/nrm.2017.77#Sec2
            warnings.warn('Mature tRNA sequence not in the expected length range (76<=L<=93)')
            
#         if anticodon_sequence != None:
#             warnings.warn('Current iteration of ME-model synthesizes a generic tRNA, not a codon-specific one')
#             if anticodon_sequence not in maturetrna_sequence:
#                 raise ValueError('Anticodon sequence not in mature tRNA sequence')
                
        if len(set(modifications.keys()).difference(allowed_trna_modifications)) > 0:
            warning_ = 'At least one of the listed modifications is not currently considered in this model'
            warnings.warn(warning_)
            modifications = {k:v for k in modifications.keys() if k in allowed_trna_modifications.keys()}
        
        if intron_sequences != None and type(intron_sequences) != list:
            raise ValueError('intron_sequences must be a list of sequences, one for each intron')


        
        self.maturetrna_sequence = maturetrna_sequence
        self.id = id_
        self.three_trailer_seq = three_trailer_seq
        self.five_leader_seq = five_leader_seq
#         self.anticodon_sequence = anticodon_sequence
        self.modifications = modifications
        self.intron_sequences = intron_sequences
        
        if self.five_leader_seq != None:
            self.pretrna_sequence = self.five_leader_seq + self.maturetrna_sequence[:-3] 
        else:
            self.pretrna_sequence = self.maturetrna_sequence[:-3]
        
        if self.intron_sequences != None: # position doesn't matter, just need # of elements and mass balance
            self.pretrna_sequence += ''.join(self.intron_sequences)
            
        if self.three_trailer_seq != None:
            self.pretrna_sequence += self.three_trailer_seq   
        
        pretrna_base_counts, trna_base_counts = dict(), dict()
        for base_letter in seq_element_map.keys():
            pretrna_base_counts[base_letter] = self.pretrna_sequence.count(base_letter)
            trna_base_counts[base_letter] = self.maturetrna_sequence.count(base_letter)
        for k,v in trna_base_counts.items():
            if v > pretrna_base_counts[k]:
                raise ValueError('Number of ' + k + ' bases in pretrna sequence less than that of trna sequence')

        self.pretrna_base_counts = pretrna_base_counts
        self.trna_base_counts = trna_base_counts


# Reactions

In [5]:
def make_trna_metabolite(name, seq, compartment = 'n', triphosphate = True):

    trna_n = cobra.Metabolite(name + '_trna[' + compartment + ']')
    trna_n.compartment = compartment
    base_counts, elements = get_base_counts_and_elements(seq, triphosphate = triphosphate) # utils function

    trna_n.elements = elements
    trna_n.charge = -len(seq)
    
    if triphosphate:
        trna_n.charge -= 3
    
    return trna_n, base_counts

def trna_fragment_degradation(fragment, fragment_base_counts, fragment_seq, name, triphosphate = True, 
                              nucleus = True):
    # exonucleolytic cleavage reaction

    fragment_degradation = cobra.Reaction(name + '_tRNA_DEGRADATION')
    fragment_degradation.subsytem = 'tRNA_Biogenesis'
    fragment_degradation.gene_reaction_rule = ' and '.join(lariat_machinery['Exosome'])

    if nucleus: 
        rxn = dict()
        rxn[h2o_n] = -sum(fragment_base_counts.values())+1
        rxn[fragment] = -1
        for k,v in nmp_map_n.items():
            rxn[v] = fragment_base_counts[k]

        # triphosphate on 5' end
        if triphosphate:
            rxn[nmp_map_n[fragment_seq[0]]] -= 1
            rxn[ntp_map_n[fragment_seq[0]]]  = 1  
            rxn[h_n] = sum(fragment_base_counts.values())-1
        else:
            rxn[h_n] = sum(fragment_base_counts.values()) # extra H on 5' end <--unsure about this

        fragment_degradation.add_metabolites(rxn)

        
    else:
        rxn = dict()
        rxn[h2o_c] = -sum(fragment_base_counts.values())+1
        rxn[fragment] = -1
        for k,v in nmp_map_c.items():
            rxn[v] = fragment_base_counts[k]

        # triphosphate on 5' end
        if triphosphate:
            rxn[nmp_map_c[fragment_seq[0]]] -= 1
            rxn[ntp_map_c[fragment_seq[0]]]  = 1  
            rxn[h_c] = sum(fragment_base_counts.values())-1
        else:
            rxn[h_c] = sum(fragment_base_counts.values()) # extra H on 5' end <--unsure about this

        fragment_degradation.add_metabolites(rxn)
        
    return fragment_degradation 

In [239]:
def transcribe_pretrna(trna_info):
    pretrna_transcript_n, pretrna_base_counts = make_trna_metabolite(trna_info.id + '_pre', trna_info.pretrna_sequence, 
                                                                     compartment = 'n', triphosphate = True)

    pretrna_transcription = cobra.Reaction('TRANSCRIPTION_PRE_TRNA_' + trna_info.id)
    pretrna_transcription.subsytem = 'tRNA_Biogenesis'
    rxn = dict()
    for ntp, base_letter in seq_metabolite_map.items():
        rxn[ntp] = -1*pretrna_base_counts[base_letter]
    rxn[ppi_n] = len(trna_info.pretrna_sequence) - 1
    rxn[pretrna_transcript_n] = 1
    pretrna_transcription.add_metabolites(rxn)
    pretrna_transcription.gene_reaction_rule = ' and '.join(rnap3_transcription_machinery)     
    return pretrna_transcription, pretrna_transcript_n

def process_trna(trna_info, pretrna_transcript_n):
    '''
    
    This reaction processes pre-tRNA into mature tRNA in the nucleus. 
    This includes: CCA synthesis, 5' leader and 3' trailer cleavage (and degradation as separate reactions),
    and splicing (and intron degradation as a separate reactions for each intron).
    
    '''
    # processing includes 5' leader and 3' trailer degradation, CCA synthesis
    # in the future, should inlcude splicing
    
    rxn = {pretrna_transcript_n: -1}
    # CCA synthesis
    rxn[ntp_map_n['C']] = -2
    rxn[ntp_map_n['A']] = -1
    rxn[ppi_n] = 3
    trna_processing_machinery = TRNT1.copy()
    
    # initialize
    reactions = list()
    rxn[h2o_n] = 0 
    
    # 5' cleavage
    if trna_info.five_leader_seq != None: # if there is a 5' leader sequence
        five_frag_n, five_frag_base_counts = make_trna_metabolite(trna_info.id + "_5'_leader_fragment", 
                                                             trna_info.five_leader_seq, compartment = 'n',
                                                                 triphosphate = True)
        rxn[five_frag_n] = 1
        rxn[h2o_n] -= 1 #endonuclolytic cleavage (RNAse P)
        trna_processing_machinery += RNASEP
        
        five_leader_degradation = trna_fragment_degradation(five_frag_n, five_frag_base_counts, trna_info.five_leader_seq, 
                              trna_info.id + "_5'_leader_fragment", triphosphate = True, 
                                  nucleus = True)
        reactions += [five_leader_degradation]
        
        tp = False 
    else:
        tp = True

    # mature tRNA
    trna_transcript_n, trna_base_counts = make_trna_metabolite(trna_info.id, trna_info.maturetrna_sequence, 
                                                              compartment = 'n', triphosphate = tp)
    rxn[trna_transcript_n] = 1

    # 3' cleavage
    if trna_info.three_trailer_seq != None:
        three_frag_n, three_frag_base_counts = make_trna_metabolite(trna_info.id + "_3'_trailer_fragment", 
                                                         trna_info.three_trailer_seq, compartment = 'n',
                                                             triphosphate = False)
        rxn[three_frag_n] = 1
        rxn[h2o_n] -= 1 #endonuclolytic cleavage (RNase Z)
        trna_processing_machinery += RNASEZ
        
        
        three_trailer_degradation = trna_fragment_degradation(three_frag_n, three_frag_base_counts, 
                                                              trna_info.three_trailer_seq, 
                             trna_info.id + "_3'_trailer_fragment", triphosphate = False, 
                                  nucleus = True)
        reactions += [three_trailer_degradation]

        

    # splicing of intron
    if trna_info.intron_sequences != None:
        n_introns = len(trna_info.intron_sequences)
        trna_introns_n = dict()
        for i in range(len(trna_info.intron_sequences)):
            trna_intron_n, trna_intron_base_counts = make_trna_metabolite(trna_info.id + "_intron_" + str(i), 
                                                             trna_info.intron_sequences[i], compartment = 'n',
                                                                 triphosphate = False)
            rxn[trna_intron_n] = 1
            trna_processing_machinery += trna_splicing_machinery
            
            intron_degradation = trna_fragment_degradation(trna_intron_n, trna_intron_base_counts, 
                                                           trna_info.intron_sequences[i],
                                                           trna_info.id + "_intron_" + str(i), 
                                                           triphosphate = False, nucleus = True)
            reactions += [intron_degradation]

        rxn[h2o_n] -= n_introns

    trna_processing = cobra.Reaction('PROCESSING_TRNA_' + trna_info.id)
    trna_processing.subsytem = 'tRNA_Biogenesis'
    trna_processing.add_metabolites(rxn)
    trna_processing.gene_reaction_rule = ' and '.join(trna_processing_machinery)

    reactions += [trna_processing]
    
    return reactions, trna_transcript_n

def modify_trna_nuclear(trna_info, trna_transcript_n):
    '''Add to the if statement in the future. This is for nuclear modifications'''
    if len(trna_info.modifications) > 0:
        raise ValueError('Modifications are not currently considered')
#         modified_trna_transcript_n = trna_transcript_n.copy()
#         modified_trna_transcript_n.id = trna_info.id + '_modified_trna[n]'
        ####
    else:
        trna_modifications_nuclear = None
        modified_trna_transcript_n = trna_transcript_n
    return trna_modifications_nuclear, modified_trna_transcript_n

def primary_export_trna(trna_info, modified_trna_transcript_n):
    trna_transcript_c = modified_trna_transcript_n.copy()
    trna_transcript_c.id = trna_transcript_c.id.replace('[n]', '[c]')
    trna_transcript_c.compartment = 'c'

    trna_primary_export = cobra.Reaction('PRIMARY_EXPORT_TRNA_' + trna_info.id)
    trna_primary_export.subsytem = 'tRNA_Biogenesis'
    trna_primary_export.name = 'trna nuclear export'
    trna_primary_export.add_metabolites({modified_trna_transcript_n: -1, trna_transcript_c: 1})
    trna_primary_export.gene_reaction_rule = 'xpot_nucleocytoplasmic_export'
    
    return trna_primary_export, trna_transcript_c

def modify_trna_cytosolic(trna_info, trna_transcript_c):
    '''Add to the if statement in the future. This is for cytosolic modifications'''
    if len(trna_info.modifications) > 0:
        raise ValueError('Modifications are not currently considered')
    else:
        trna_modifications_cytosolic = None
        modified_trna_transcript_c = trna_transcript_c
    return trna_modifications_cytosolic, modified_trna_transcript_c

def degrade_trna(trna_info, modified_trna_transcript_c):
    # currently built on the assumption that there are no post-transcriptional modifications on trna 
    
    if trna_info.five_leader_seq != None: # if there is a 5' leader sequence
        tp = False 
    else:
        tp = True
    trna_degradation = trna_fragment_degradation(fragment = modified_trna_transcript_c, 
                                                 fragment_base_counts = trna_info.trna_base_counts, 
                                                 fragment_seq = trna_info.maturetrna_sequence, 
                                                 name = trna_info.id, triphosphate = tp,
                                                 nucleus = False)
    trna_degradation.gene_reaction_rule = XRN1[0]
    return trna_degradation
    

def charge_trna(trna_info, modified_trna_transcript_c):
    '''tRNA charging reaction combines the activation and charging steps into one reaction.'''
    
    
    # in the future, this should take into account anticodon sequence, which should be in 
    # trna_info id
    
    # now, since just a generic charging reaction, will create one for each amino acid (for loop)
    
    # diagram: https://www.researchgate.net/figure/The-reaction-scheme-for-the-two-steps-of-aminoacylation-reaction-at-the-active-site-of_fig4_231225238
    trna_charging_reactions, charged_trna_metabolites = [], []
    for code, aa in seq_amino_acid_map_c.items():
        elements = modified_trna_transcript_c.elements
        # attachment - loss of O-H on tRNA
        elements['H'] -= 1 
        elements['O'] -= 1
        for element, count in aa.elements.items():
            if element in elements.keys():
                elements[element] += count
            else:
                elements[element] = count

        charged_trna_c = cobra.Metabolite('charged_' + trna_info.id + '_' + code + '_trna[c]')
        charged_trna_c.compartment = 'c'
        charged_trna_c.elements = elements
        # +1 for loss of negative charge on oxygen of amino acid
        charged_trna_c.charge = modified_trna_transcript_c.charge + aa.charge + 1 

        trna_charging = cobra.Reaction('CHARGING_TRNA_' + trna_info.id + '_' + code)
        trna_charging.subsytem = 'tRNA_Biogenesis'
        rxn = {modified_trna_transcript_c: -1, aa: -1, charged_trna_c: 1, atp_c: -1, ppi_c: 1, amp_c: 1}
        trna_charging.add_metabolites(rxn)
        # add gprs
        genes = seq_synthetase_map[code]
        if len(genes) == 1:
            trna_charging.gene_reaction_rule = genes[0]
        else:
            trna_charging.gene_reaction_rule = ' and '.join(genes)
        

        trna_charging_reactions.append(trna_charging)
        charged_trna_metabolites.append(charged_trna_c)
    return trna_charging_reactions, charged_trna_metabolites
    

def trna_biogenesis(trna_info):
    '''trna_info is an object of class trna_information'''
    pretrna_transcription, pretrna_transcript_n = transcribe_pretrna(trna_info)
    trna_processing_reactions, trna_transcript_n = process_trna(trna_info, pretrna_transcript_n)
    # right now, no modification reaction and modified and non-modified are same metabolite
    trna_modifications_nuclear, modified_trna_transcript_n = modify_trna(trna_info, trna_transcript_n) 
    trna_primary_export, trna_transcript_c = primary_export_trna(trna_info, modified_trna_transcript_n)
    trna_modifications_cytosolic, modified_trna_transcript_c = modify_trna_cytosolic(trna_info, trna_transcript_c)
    trna_degradation = degrade_trna(trna_info, modified_trna_transcript_c)
    trna_charging_reactions, charged_trna_metabolites = charge_trna(trna_info, modified_trna_transcript_c)
    #---------------------------------------------------------------------------------------------------
    
    reactions = [pretrna_transcription] + trna_processing_reactions + [trna_primary_export, trna_degradation] 
    reactions += trna_charging_reactions
    if trna_modifications_nuclear != None:
        reactions += [trna_modifications_nuclear]
    if trna_modifications_cytosolic != None:
        reactions += [trna_modifications_cytosolic]
    
    # reactiosn will be added to model, both charged and uncharged (modified) trna will be involved in translationr reactions
    return reactions, charged_trna_metabolites, modified_trna_transcript_c
     

# Consensus Sequences

positionally-independent (position won't effect .elements of cobra.Metabolite in cobra.Reaction)

In [ ]:
def get_base_frequency(seq_col, L):
    base_counts = {'T': 0, 'C': 0, 'G': 0, 'A': 0}
    for seq in trna_data[seq_col]:
        for base in base_counts.keys():
            base_counts[base] += seq.count(base)
    total = sum(base_counts.values())   
    base_frequencies = {base: (counts/total) for base,counts in base_counts.items()}
    base_frequencies['U'] = base_frequencies['T']
    base_frequencies.pop('T')

    final_seq = ''
    for base, frequency in base_frequencies.items():
        final_seq += base*round(frequency*L)

    # this assumes that if lengths are off, they are only off by one (accurate assumption for this dataset)
    if len(final_seq) != L:
        base_counts = {base: final_seq.count(base) for base in base_frequencies.keys()}
        if len(final_seq) > L:
            final_seq = ''
            to_remove = [k for k,v in base_frequencies.items() if v == min(base_frequencies.values())][0]
            base_counts[to_remove] -= 1
            for base in base_counts.keys():
                final_seq += base*base_counts[base]
        else:
            final_seq = ''
            to_add = [k for k,v in base_frequencies.items() if v == max(base_frequencies.values())][0]
            base_counts[to_add] += 1
            for base in base_counts.keys():
                final_seq += base*base_counts[base]

    return final_seq

trna_data = pd.read_excel(local_data_path + 'raw/trna_leaders_and_trailers.xlsx')
trna_data['Mature_Length'] = trna_data['mature seq'].apply(lambda x: len(x))

L_mature = trna_data.Mature_Length.value_counts()[trna_data.Mature_Length.value_counts() == trna_data.Mature_Length.value_counts().max()].index.tolist()[0]
L_leader = trna_data[' leader length'].value_counts()[trna_data[' leader length'].value_counts() == trna_data[' leader length'].value_counts().max()].index.tolist()[0]
L_trailer = trna_data[' trailer length'].value_counts()[trna_data[' trailer length'].value_counts() == trna_data[' trailer length'].value_counts().max()].index.tolist()[0]

mature_seq = get_base_frequency('mature seq', L_mature)
leader_seq, trailer_seq = get_base_frequency('leader seq', L_leader), get_base_frequency('trailer seq', L_trailer)

# CCA tail (position matters in the code)
base_counts = {base: mature_seq.count(base) for base in ['U', 'C', 'G', 'A']}
base_counts['C'] -= 2
base_counts['A'] -= 1
mature_seq = ''
for base in base_counts.keys():
    mature_seq += base*base_counts[base]
mature_seq += 'CCA'

# Generate reactions

In [240]:
trna_info = trna_information(maturetrna_sequence = mature_seq , id_ = 'generic', three_trailer_seq = trailer_seq, 
                             five_leader_seq = leader_seq, modifications = {},
                             intron_sequences = None)
trna_biogenesis_reactions, charged_trna_metabolites, modified_trna_transcript_c = trna_biogenesis(trna_info)

/Users/joycebaghdassarian/opt/anaconda3/envs/human_me/lib/python3.6/site-packages/ipykernel_launcher.py:22 UserWarning: Mature tRNA sequence not in the expected length range (76<=L<=93)
